# GRPO 正式训练：Qwen2.5-3B-Instruct + bf16 LoRA + vLLM on MedQA

试跑结论见 `results_summary/grpo_pilot_bf16_vllm/README.md`：A100 上 bf16 LoRA + vLLM 每步约 8.6 秒（回答上限 512）；**不要用 QLoRA + vLLM**（梯度会逐步消失）。

按顺序运行：检查 GPU → 设置变量 → 拉取 repo → 安装依赖 → 挂载 Drive →（可选）HF_TOKEN → 单元测试 → **训练** → 看结果

运行前：菜单 **运行时 → 更改运行时类型 → A100 GPU**。

**所有输出（checkpoint、metrics.csv、train_log.txt、最终 adapter）直接写在 Google Drive 的 `OUTPUT_DIR`**，断开连接也不会丢。

**断线 / 运行时被回收后怎么续跑**：从第 1 格重新运行到第 7 格（会重新装依赖），然后**原样**运行第 8 格。命令里带 `--resume`，会从 `OUTPUT_DIR/checkpoints` 里最新的 checkpoint 继续，日志里会显示 `[续跑] 找到 checkpoint`。
⚠️ 续跑时**不要改第 2 格的任何参数**（题数、epoch、seed、超参数），否则学习率计划和数据顺序会对不上。

In [ ]:
# 1. 检查 GPU 和 Python 版本
!nvidia-smi
import sys, torch
print('Python', sys.version.split()[0])
print('torch', torch.__version__, '| CUDA 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print('GPU:', torch.cuda.get_device_name(0), f'| compute {major}.{minor} →', 'bf16' if major >= 8 else 'fp16')
assert torch.cuda.is_available(), '没有 GPU：运行时 → 更改运行时类型 → GPU'
assert sys.version_info >= (3, 11), 'TRL 1.16 需要 Python >= 3.11'

In [ ]:
# 2. 设置变量（只需要改这里；开始训练后不要再改，否则续跑会对不上）
REPO_URL = 'https://github.com/skjdsk/medqa-calibration.git'
BRANCH   = 'main'
REPO_DIR = '/content/medqa-calibration'

TRL_VERSION = '1.16.0'

# 训练数据：MedQA train split 用 SEED 打乱后取前 N_QUESTIONS 题（0 = 全部 10178 题）
# ⚠️ 和 SFT / DPO 用同一批题（同样的 seed 和题数），比较才公平
N_QUESTIONS           = 2000   # ← 和队友确认后再改
SEED                  = 42
NUM_EPOCHS            = 1

NUM_GENERATIONS       = 4
MAX_COMPLETION_LENGTH = 1024   # 试跑用 512 时约 9% 回答被截断
LEARNING_RATE         = 1e-5
BETA                  = 0.04   # KL 系数
TEMPERATURE           = 1.0
PER_DEVICE_BS         = 4      # 爆显存就改成 2，同时 GRAD_ACCUM 改成 8（每步回答数不变）
GRAD_ACCUM            = 4      # PER_DEVICE_BS × GRAD_ACCUM 必须是 NUM_GENERATIONS 的倍数
LORA_R, LORA_ALPHA    = 16, 32
LORA_TARGET_MODULES   = 'all-linear'
USE_VLLM              = True
QUANTIZATION          = 'none' # bf16 + LoRA；不要和 vLLM 一起用 '4bit'
SAVE_STEPS            = 50     # 每 50 步存一次 checkpoint（含优化器状态，约 0.4 GB；只保留最近 2 个）

# 输出目录在 Drive 上，名字由设置决定：同样的设置 → 同一个目录 → 续跑能找到 checkpoint
EXP_NAME   = f"grpo_n{N_QUESTIONS or 'all'}_ep{NUM_EPOCHS}_len{MAX_COMPLETION_LENGTH}_seed{SEED}_{'qlora' if QUANTIZATION == '4bit' else 'bf16'}"
OUTPUT_DIR = f'/content/drive/MyDrive/medqa-calibration/grpo/{EXP_NAME}'

assert (PER_DEVICE_BS * GRAD_ACCUM) % NUM_GENERATIONS == 0, 'PER_DEVICE_BS × GRAD_ACCUM 必须能被 NUM_GENERATIONS 整除'
prompts_per_step = PER_DEVICE_BS * GRAD_ACCUM // NUM_GENERATIONS
import math
n = N_QUESTIONS or 10178
print('每步', PER_DEVICE_BS * GRAD_ACCUM, '条回答 =', prompts_per_step, '题 ×', NUM_GENERATIONS, '个回答')
print('总步数 ≈', math.ceil(n * NUM_EPOCHS / prompts_per_step))
print('输出目录：', OUTPUT_DIR)

In [ ]:
# 3. git clone / pull repo
# 私有 repo：在 Colab 左侧 🔑（Secrets）里添加 GITHUB_TOKEN（GitHub 个人访问令牌），并打开"笔记本访问"
import os, subprocess
from pathlib import Path

def sh(cmd, shown=None):
    print('$', shown or cmd)
    subprocess.run(cmd, shell=True, check=True)

clone_url = REPO_URL
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
    if gh_token and REPO_URL.startswith('https://github.com/'):
        clone_url = REPO_URL.replace('https://', f'https://{gh_token}@')
except Exception:
    pass  # 没设置 GITHUB_TOKEN 就按公开 repo 处理

if (Path(REPO_DIR) / '.git').exists():
    sh(f'git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {clone_url} {REPO_DIR}', shown=f'git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}')
os.chdir(REPO_DIR)
sh('git log -1 --oneline')
for f in ['train/grpo/grpo_data.py', 'train/grpo/rewards.py', 'train/grpo/test_rewards.py', 'train/grpo/grpo.py', 'train/grpo/estimate.py']:
    assert Path(f).exists(), f'repo 里找不到 {f}：确认文件已经 push 到 {BRANCH}'
print('cwd =', os.getcwd())

In [ ]:
# 4. 安装依赖
from pathlib import Path
ip = get_ipython()
if Path('requirements.txt').exists():
    ip.run_line_magic('pip', 'install -q -r requirements.txt')
ip.run_line_magic('pip', f'install -q "trl[peft,quantization]=={TRL_VERSION}" "matplotlib" "pandas" "pytest"')
if USE_VLLM:
    # vLLM 会安装它自己需要的 torch 版本，装完后通常要 运行时 → 重启会话，再从"设置变量"那格重新运行
    # vLLM 0.28+ 删掉了 bitsandbytes 量化，而 TRL 在 QLoRA（4-bit）时会让 vLLM 用 bitsandbytes 加载模型
    # → 固定 0.27.1（最后一个支持 bitsandbytes 的版本，且在 TRL 1.16 允许的 0.22–0.31 范围内，要求 torch==2.13.0）
    ip.run_line_magic('pip', f'install -q "trl[vllm]=={TRL_VERSION}" "vllm==0.27.1"')

# Colab 预装的 torchao 0.10 太旧：peft 0.21 在不量化（--quantization none）加 LoRA 时会直接报错
# "only versions above 0.16.0 are supported"。我们用不到 torchao（vllm / peft 都不依赖它），直接卸载
ip.system('pip uninstall -y -q torchao')

import importlib.metadata as md
for pkg in ['torch', 'transformers', 'trl', 'peft', 'bitsandbytes', 'accelerate', 'datasets'] + (['vllm'] if USE_VLLM else []):
    try:
        print(f'{pkg:14s}', md.version(pkg))
    except md.PackageNotFoundError:
        print(f'{pkg:14s} 未安装')
print('\n如果上面 pip 提示需要重启（或之后 import 报错）：运行时 → 重启会话，然后从"设置变量"那格重新运行。')

In [ ]:
# 5. 挂载 Google Drive，检查输出目录里有没有之前的 checkpoint
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('输出目录：', OUTPUT_DIR)
ckpts = sorted(Path(OUTPUT_DIR, 'checkpoints').glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
if ckpts:
    print('已有 checkpoint：', [p.name for p in ckpts], '→ 第 8 格会从', ckpts[-1].name, '继续')
else:
    print('还没有 checkpoint → 第 8 格会从头开始训练')
if Path(OUTPUT_DIR, 'adapter').exists():
    print('⚠️ 这个目录里已经有训练完的 adapter：这组设置可能已经跑完了，再跑会覆盖')

In [ ]:
# 6.（可选）设置 HF_TOKEN：Qwen2.5 和 MedQA 都是公开的，不设也能下载；设置后下载更稳定、不容易被限速
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('已设置 HF_TOKEN')
except Exception:
    print('没有找到 HF_TOKEN（可选），继续匿名下载')

In [ ]:
# 7. 跑打分函数的单元测试（几秒钟，不用 GPU）
!python -m pytest -q train/grpo/test_rewards.py

In [ ]:
# 8. 正式训练（结果直接写到 Drive 的 OUTPUT_DIR；断线后原样重跑这一格即可续跑）
args = [
    f'--n_questions {N_QUESTIONS}', f'--seed {SEED}', f'--num_train_epochs {NUM_EPOCHS}',
    f'--num_generations {NUM_GENERATIONS}', f'--max_completion_length {MAX_COMPLETION_LENGTH}',
    f'--learning_rate {LEARNING_RATE}', f'--beta {BETA}', f'--temperature {TEMPERATURE}',
    f'--per_device_train_batch_size {PER_DEVICE_BS}', f'--gradient_accumulation_steps {GRAD_ACCUM}',
    f'--lora_r {LORA_R}', f'--lora_alpha {LORA_ALPHA}', f'--lora_target_modules {LORA_TARGET_MODULES}',
    f'--quantization {QUANTIZATION}', f'--output_dir {OUTPUT_DIR}',
    f'--save_steps {SAVE_STEPS}', '--resume',
]
if USE_VLLM:
    args.append('--use_vllm')
ARGS = ' '.join(args)
print('python -m train.grpo.grpo', ARGS)
# tee -a：续跑时把日志接在后面，不覆盖之前的
!python -u -m train.grpo.grpo {ARGS} 2>&1 | tee -a {OUTPUT_DIR}/train_log.txt

**前 10 步要看的**（确认没问题再离开）：
- `mem`：没有爆显存
- 每步耗时 × 总步数 ≈ 总时长
- `sampling/importance_sampling_ratio/mean` 接近 1，`sampling/sampling_logp_difference/mean` 约 0.02 且不上涨
- `grad_norm` 大约 0.05–0.3（偶尔接近 0 是因为那一步组内全对或全错）
- `截断` 比试跑的 0.09 低

**训练中**（`OUTPUT_DIR/metrics.csv` 每步写一行，可以直接在 Drive 里看）：reward 大体向上、格式正确率保持 0.9 左右、KL 慢慢上升但不超过约 0.1、回答长度没有一直变长或突然变短。

In [ ]:
# 9. 看结果（训练结束后运行。训练中想看进度，直接在 Drive 里打开 OUTPUT_DIR/metrics.csv，
#    不要停止第 8 格：停止后虽然能续跑，但会丢掉最近一个 checkpoint 之后的步数）
import json
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(f'{OUTPUT_DIR}/metrics.csv')
cols = ['step', 'step_time_s', 'peak_mem_allocated_gb', 'reward_mean', 'frac_groups_all_same',
        'format_rate', 'mean_completion_tokens', 'truncated_frac', 'kl', 'grad_norm']
print(f'已训练 {int(df.step.max())} 步')
# 每 50 步取平均，看趋势
df['block'] = (df.step - 1) // 50
display(df.groupby('block')[[c for c in cols if c in df.columns and c != 'step']].mean().round(4))

report_path = Path(OUTPUT_DIR, 'pilot_report.json')
if report_path.exists():  # 训练结束后才有
    report = json.load(open(report_path))
    print('平均每步耗时（秒）:', report['timing']['avg_step_time_s_excl_warmup'])
    print('显存峰值（GB）:', report['memory']['peak_mem_allocated_gb_max_over_steps'])
    print('reward:', report['reward'])
    print('格式正确率:', report['format_rate'])
    print('稳定性:', report['stability'])
    display(Image(f'{OUTPUT_DIR}/curves.png'))
    print('LoRA adapter：', Path(OUTPUT_DIR, 'adapter'))